<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/assignment3_661_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 3: Image Classification with CNNs (CIFAR-100 Coarse Labels)

---

### Dataset: CIFAR-100 (Coarse Labels)

The CIFAR-100 (coarse) dataset contains **60,000 color images** across **20 broad superclasses**.

- Each image is:
  - Size: **32 x 32 pixels**
  - Channels: **RGB (3 channels)**

- Dataset Composition:
  - **50,000 training images**
  - **10,000 test images**
  - 20 coarse classes (superclasses)

- **Classes (coarse superclasses):**
  0. aquatic mammals
  1. fish
  2. flowers
  3. food containers
  4. fruit and vegetables
  5. household electrical devices
  6. household furniture
  7. insects
  8. large carnivores
  9. large man-made outdoor things
  10. large natural outdoor scenes
  11. large omnivores and herbivores
  12. medium-sized mammals
  13. non-insect invertebrates
  14. people
  15. reptiles
  16. small mammals
  17. trees
  18. vehicles 1
  19. vehicles 2

This notebook builds on the Week 10 CNN notebook with minimal edits: same architecture, same training setup, just adapted to CIFAR-100 coarse (20 classes instead of 10).

## **Step 1: Load and Explore the dataset**

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import random

Setting a random seed of 14 so the results are reproducible across reruns. This is especially important here because the notebook trains three different models (baseline, no-normalization, and tuned), and I want the comparisons between them to be fair.

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(14)
np.random.seed(14)
random.seed(14)

In [ ]:
from tensorflow.keras import datasets, utils, layers, models, losses, optimizers

# Load CIFAR-100 (coarse labels = 20 superclasses)
(x_train, y_train), (x_test, y_test) = datasets.cifar100.load_data(label_mode='coarse')

NUM_CLASSES = 20

Swapped the CIFAR-10 loader from W10 for CIFAR-100 with `label_mode='coarse'` and bumped `NUM_CLASSES` from 10 to 20.

In [ ]:
# Check the shape of the images
print("Training data shape:", x_train.shape)  # (50000, 32, 32, 3)
print("Single image shape:", x_train[0].shape)  # (32, 32, 3)

## **Step 2: Normalize the Data and One-Hot Encode the Labels**

In [ ]:
# Normalize pixel values from 0-255 to 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

**One-Hot Encoding the Labels**

CIFAR-100 coarse labels come as integers from 0 to 19 (one per superclass). Converting them to one-hot vectors so they line up with the `categorical_crossentropy` loss function. Same approach as W10 just with 20 classes instead of 10.

In [ ]:
# one-hot encode the labels
y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_test = utils.to_categorical(y_test, NUM_CLASSES)

## **Step 3: Model Building (Functional API) and Compilation**

Using the same CNN architecture from Week 10 without any changes:

-  **Conv2D**: Convolution layers with 3×3 kernels to learn local patterns.
-  **Strides = 2**: Used instead of pooling to reduce the spatial dimensions (downsampling).
-  **Padding = "same"**: Keeps output dimensions consistent.
-  **BatchNormalization**: Normalizes layer outputs to stabilize and accelerate training.
-  **LeakyReLU**: Activation function that avoids "dead neurons" (improves flow of gradients).
-  **Dropout**: Regularizes the model to prevent overfitting.
-  **Dense + Softmax**: Final layers, now classifying into 20 CIFAR-100 coarse categories instead of 10.

In [ ]:
input_layer = layers.Input(shape=(32, 32, 3))

# Convolution Block 1 (no downsampling yet):
x = layers.Conv2D(filters=32, kernel_size=3, strides=1, padding="same")(input_layer)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 2 (with downsampling):
x = layers.Conv2D(filters=32, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 3:
x = layers.Conv2D(filters=64, kernel_size=3, strides=1, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 4 (with downsampling):
x = layers.Conv2D(filters=64, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Dense layers:
x = layers.Flatten()(x)
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(rate=0.5)(x)

# Output layer (20 classes for CIFAR-100 coarse)
output_layer = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(input_layer, output_layer)

In [ ]:
model.summary()

**Compiling the Model**

- **Optimizer**: adam
- **Loss**: categorical_crossentropy
- **Metric**: accuracy

Same compile settings as W10.

In [ ]:
from tensorflow.keras.optimizers import Adam

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## **Step 4: Training and Evaluating the Model**

Training for 20 epochs with a 20% validation split and shuffle enabled. W10 used 10 epochs but CIFAR-100 coarse is a harder problem (20 classes vs 10) so I bumped it up to 20 to give the model more time to converge. Everything else matches W10.

**Training Parameters:**

- Epochs → `20`
- Batch size → `32`
- Validation split → `0.2` (20% of training data used for validation)

In [ ]:
# train the model
history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=20,
    shuffle=True,
    validation_split=0.2,
)

**Evaluate the Model**

In [ ]:
# accuracy and loss plots

def plot_training_history(history, title_prefix=""):
    plt.figure(figsize=(12, 4))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f"{title_prefix}Training vs. Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f"{title_prefix}Training vs. Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_history(history, title_prefix="Baseline: ")

In [ ]:
# Final test evaluation (once)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f} | Test loss:  {test_loss:.3f}")

## **Step 5: Make Predictions and Confusion Matrix**

Using the 20 coarse class names from the assignment instructions for readability in the prediction samples and the confusion matrix.

In [ ]:
CLASSES = np.array(
    [
        "aquatic mammals",
        "fish",
        "flowers",
        "food containers",
        "fruit and vegetables",
        "household electrical devices",
        "household furniture",
        "insects",
        "large carnivores",
        "large man-made outdoor things",
        "large natural outdoor scenes",
        "large omnivores and herbivores",
        "medium-sized mammals",
        "non-insect invertebrates",
        "people",
        "reptiles",
        "small mammals",
        "trees",
        "vehicles 1",
        "vehicles 2",
    ]
)

preds = model.predict(x_test)
preds_single = CLASSES[np.argmax(preds, axis=-1)]
actual_single = CLASSES[np.argmax(y_test, axis=-1)]

In [ ]:
n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx]
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=8,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=8,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(x_test), axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(14, 12))  # larger figure for 20x20 matrix
disp.plot(ax=ax, xticks_rotation=90, colorbar=True)
plt.title("Confusion Matrix (CIFAR-100 Coarse)")
plt.tight_layout()
plt.show()

## **Step 6: Experiment — Skip Normalization (Part 2 Q4)**

Retraining the exact same architecture on raw pixel values (0-255) instead of normalized ones (0-1) to see what happens when normalization is skipped. Same 20 epochs, same validation split, same seed so the comparison is fair.

In [ ]:
# Reload the raw data (intentionally skipping the /255.0 normalization step)
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = datasets.cifar100.load_data(label_mode='coarse')

# Still need to one-hot encode the labels for categorical_crossentropy
y_train_raw = utils.to_categorical(y_train_raw, NUM_CLASSES)
y_test_raw = utils.to_categorical(y_test_raw, NUM_CLASSES)

print("Raw pixel range:", x_train_raw.min(), "to", x_train_raw.max())

In [ ]:
# Reset the seed so this run is reproducible on its own
tf.random.set_seed(14)
np.random.seed(14)
random.seed(14)

# Build the same architecture as the baseline (identical layer-by-layer)
input_layer_nn = layers.Input(shape=(32, 32, 3))

x = layers.Conv2D(filters=32, kernel_size=3, strides=1, padding="same")(input_layer_nn)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(filters=32, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(filters=64, kernel_size=3, strides=1, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

x = layers.Conv2D(filters=64, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

x = layers.Flatten()(x)
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(rate=0.5)(x)

output_layer_nn = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_no_norm = models.Model(input_layer_nn, output_layer_nn)

model_no_norm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train on raw (non-normalized) pixel values
history_no_norm = model_no_norm.fit(
    x_train_raw,
    y_train_raw,
    batch_size=32,
    epochs=20,
    shuffle=True,
    validation_split=0.2,
)

In [ ]:
plot_training_history(history_no_norm, title_prefix="No Normalization: ")

test_loss_nn, test_acc_nn = model_no_norm.evaluate(x_test_raw, y_test_raw, verbose=0)
print(f"No-normalization test accuracy: {test_acc_nn:.3f} | test loss: {test_loss_nn:.3f}")

## **Step 7: Experiment — Tuning (Doubled Filters) (Part 2 Q5)**

For the tuning experiment I doubled the filter counts in all four convolutional blocks (32 → 64 and 64 → 128). CIFAR-100 coarse has 20 classes (double CIFAR-10), so giving the model more filters should give it more feature detectors to distinguish between the extra classes. Everything else stays the same — same 20 epochs, same seed, same validation split — so any difference in results comes from the filter change alone.

In [ ]:
# Reset the seed so this run is reproducible on its own
tf.random.set_seed(14)
np.random.seed(14)
random.seed(14)

# Build the tuned model (same architecture as baseline but with doubled filters)
input_layer_t = layers.Input(shape=(32, 32, 3))

# Convolution Block 1 — doubled filters (32 -> 64)
x = layers.Conv2D(filters=64, kernel_size=3, strides=1, padding="same")(input_layer_t)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 2 — doubled filters (32 -> 64)
x = layers.Conv2D(filters=64, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 3 — doubled filters (64 -> 128)
x = layers.Conv2D(filters=128, kernel_size=3, strides=1, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 4 — doubled filters (64 -> 128)
x = layers.Conv2D(filters=128, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

x = layers.Flatten()(x)
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(rate=0.5)(x)

output_layer_t = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_tuned = models.Model(input_layer_t, output_layer_t)

model_tuned.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_tuned.summary()

In [ ]:
# Train the tuned model on the normalized data
history_tuned = model_tuned.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=20,
    shuffle=True,
    validation_split=0.2,
)

In [ ]:
plot_training_history(history_tuned, title_prefix="Tuned (Doubled Filters): ")

test_loss_t, test_acc_t = model_tuned.evaluate(x_test, y_test, verbose=0)
print(f"Tuned model (doubled filters) test accuracy: {test_acc_t:.3f} | test loss: {test_loss_t:.3f}")

### Summary Comparison

Side-by-side test results for all three runs. Numbers from here will feed directly into the Part 2 report.

In [ ]:
print("=" * 65)
print(f"{'Model':<40} {'Test Acc':>10} {'Test Loss':>10}")
print("=" * 65)
print(f"{'Baseline (W10 arch, normalized)':<40} {test_acc:>10.3f} {test_loss:>10.3f}")
print(f"{'No normalization':<40} {test_acc_nn:>10.3f} {test_loss_nn:>10.3f}")
print(f"{'Tuned (doubled filters)':<40} {test_acc_t:>10.3f} {test_loss_t:>10.3f}")
print("=" * 65)